# 09 — Product Policy、Scenario 与 Execution V2

这是 Q20 的可执行端到端教程。它从产品层 `RuleBuilder` / `PolicyBuilder` 开始，经过统一的 `fg.query(...).bind(...).select(...).plan(...).run()`，展示确定性与 ProbLog 点概率执行、Scenario V2、`WeightedChoice`、结构化 Result / Explain 数据和 detached replay。

它刻意不手写 `PolicyAll`、`PolicyAny`、address 或底层引擎参数：这些是 compiler/application 层实现细节。业务、Agent 和 Meander 应构造这里所示的声明式资产、Query、Scenario 和 profile。

> 每个代码 cell 都调用真实 FactGraph API 并断言结果。Scenario 只构造一次运行的 effective world，绝不会修改源 ledger。

## 运行前提与边界

从仓库根目录或 `examples/` 目录，使用 `factpy` Jupyter kernel 运行。ProbLog cell 使用真正的 ProbLog adapter；如果本地没有它，不能把它替换成 Native 并宣称是概率结果。

V2 的 ProbLog profile 固定记录 `problog`、`native`、`souffle` 三个 engine pins，但目前只有 `problog` 执行点概率模型；Native 和 Soufflé 会以显式 `unsupported` frame 返回。它们不是零概率、false，或隐式 fallback。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / 'src').exists() and (_repo_root.parent / 'src').exists():
    _repo_root = _repo_root.parent
_src_dir = _repo_root / 'src'
if str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))

from factgraph.sdk import (
    AssetMeta,
    Entity,
    EntityRef,
    Field,
    Identity,
    SDKStore,
    vars,
    outcome_from_run_v2,
)


## 1. Schema、受控数据和产品 Rule asset

`AssetMeta` 是单独封存的描述性资产数据：它可携带 name、description、tags，但不会改变 Rule/Policy 的逻辑 compiler digest。`asset_binding_digest` 则把这份 descriptor 与它的精确逻辑目标绑定，供 sealed V2 Run 捕获。

`fg.rule_builder(...).build(...)` 是 staged form；相同参数也可以直接给 `fg.build_rule(...)`。本 cell 实际执行两种形式。Rule body 使用 SDK `vars(...)` / `Entity(var).field` DSL，`semantic_ports` 直接使用 `Person` 与 `Person.age` 这样的 SDK schema descriptor。二者都不注册一个全局 Rule 名称。

In [2]:
class Person(Entity):
    person_id: str = Identity()
    age: int = Field()


fg = SDKStore([Person])
alice_e_ref = fg.entities.create(Person, person_id='alice')
bob_e_ref = fg.entities.create(Person, person_id='bob')
fg.fields.set(Person.age, alice_e_ref, 30)
fg.fields.set(Person.age, bob_e_ref, 20)

# Query bindings are typed EntityRef values. Scenario accepts the managed e_ref string.
alice = EntityRef('Person', {'person_id': 'alice'})
bob = EntityRef('Person', {'person_id': 'bob'})

with vars('person', 'age') as (person, age):
    person_values = (
        fg.rule_builder(
            'person_values',
            version='1',
            meta=AssetMeta(
                name='Person values',
                description='Expose a person and its current age',
                tags=('demo', 'people'),
            ),
        )
        .build(
            when=(Person(person), Person(person).age == age),
            ports={'person': person, 'age': age},
            semantic_ports={'person': Person, 'age': Person.age},
        )
    )

# Direct form: same product boundary, not a second compiler or a registry write.
with vars('direct_person', 'direct_age') as (direct_person, direct_age):
    direct_person_values = fg.build_rule(
        id='person_values_direct',
        version='1',
        meta=AssetMeta(name='Direct person values', tags=('demo',)),
        when=(Person(direct_person), Person(direct_person).age == direct_age),
        ports={'person': direct_person, 'age': direct_age},
        semantic_ports={'person': Person, 'age': Person.age},
    )

assert person_values.asset_meta.name == 'Person values'
assert person_values.asset_snapshot()['descriptor_digest'] == person_values.asset_meta.descriptor_digest
assert direct_person_values.asset_meta.name == 'Direct person values'
assert direct_person_values.contract.ports['person'].endpoint.entity_type == 'Person'
assert fg.fields.get(Person.age, alice_e_ref) == 30


## 2. PolicyBuilder：局部 occurrence、字面量比较、field navigation 与嵌套逻辑

`use(rule, as_='...')` 创建的是此 Policy 内局部且 owner-bound 的 occurrence handle，不是注册动作。`all(...)` / `any(...)` 保留 authored topology；它们始终是确定性逻辑节点。`fg.build_policy(..., build=lambda p: ...)` 是相同构造方式的 direct form；callback 只取得它自己那一个 builder 的 handles。

端口 handle 支持受限且带类型的表达：`older.age > 12`、`older.person.field('age') > younger.person.field('age')`。实体合一要使用 `builder.same(...)`，而不是 Python `==`；Python `and` / `or`、链式比较和隐式 entity equality 都会失败而非悄悄改变逻辑。

In [3]:
ranked = fg.policy_builder(
    'ranked_people',
    version='1',
    meta=AssetMeta(
        name='Rank eligible people',
        description='Compare eligible people by age',
        tags=('demo', 'ranking'),
    ),
)
older = ranked.use(person_values, as_='older')
younger = ranked.use(person_values, as_='younger')
left_gate = ranked.use(person_values, as_='left_gate')
right_gate = ranked.use(person_values, as_='right_gate')

ranked_policy = ranked.build(
    ranked.all(
        older,
        younger,
        older.age > 12,
        older.person.field('age') > younger.person.field('age'),
        ranked.any(ranked.all(left_gate), ranked.all(right_gate)),
    )
)

# Direct Policy form: its callback receives a fresh builder; `people` is local to it.
direct_policy = fg.build_policy(
    id='direct_people',
    version='1',
    meta=AssetMeta(name='Direct people'),
    build=lambda p: p.all(p.use(direct_person_values, as_='people')),
)

assert ranked_policy.policy.id == 'ranked_people'
assert ranked_policy.asset_meta.tags == ('demo', 'ranking')
assert ranked_policy.requires_v2_profile is False
assert direct_policy.policy.id == 'direct_people'
assert direct_policy.asset_meta.name == 'Direct people'
assert direct_policy.requires_v2_profile is False


## 3. Deterministic V2 profile：同一个 Query 入口

V2 profile 是 immutable、target-scoped 的 execution declaration。它不接受旧式通用 `config` 或任意 `engine_options` bag。确定性 profile 只 pin Native；没有 Scenario 时 `.plan(profile=...)` 仍会创建一个显式的 no-overlay V2 world。

In [4]:
native_profile = (
    fg.execution.native_deterministic(
        target=ranked_policy,
        name='ranked-native-v2',
        max_rows=10,
    )
    .build()
)

native_run = (
    fg.query(ranked_policy)
    .bind(older.person, alice)
    .bind(younger.person, bob)
    .select('older_age', older.age)
    .plan(profile=native_profile)
    .run()
)

native_frame = native_run.effective.engine_frames[0]
assert (native_frame.engine, native_frame.status) == ('native', 'succeeded')
assert len(native_frame.observations) == 1
assert dict(native_frame.observations[0].values)['older_age'].value == 30
# These are deliberately separately sealed *named* worlds, even without an overlay.
assert native_run.baseline.world_capture_digest != native_run.effective.world_capture_digest
assert native_run.baseline.engine_frames[0].observations == native_frame.observations


## 4. 结构化 Result / Explain：数据先于文本

`.run()` 返回 durable replay carrier `EvaluationRunV2`；`outcome_from_run_v2(...)` 在 SDK 层打开面向 UI / Agent 的只读结构化视图。调用者必须显式选择一条 observation，不能隐式取第一行。

这里的 Native V2 runtime 尚未捕获 detached EvidenceGraph，因此 Explain 数据明确给出 `not_available`，而不是制造一个图。`repr` 只用于诊断，`render_text()` / `narrate()` 仍是纯展示 helper；业务代码应读取结构化字段，绝不解析 prose。

In [5]:
native_outcome = outcome_from_run_v2(native_run)
native_result = native_outcome.effective
native_row = native_result.rows[0]
native_explain = native_outcome.explain(native_row)

assert native_result.source_protocol == 'evaluation_run_v2'
assert native_row.point_probability is None
assert native_explain.evidence.state == 'not_available'
assert native_explain.evidence.reason_code == 'NATIVE_V2_DETACHED_EVIDENCE_GRAPH_NOT_IMPLEMENTED'

{
    'engine': native_result.engine,
    'row_values': {value.alias: value.value for value in native_row.values},
    'evidence_state': native_explain.evidence.state,
}


{'engine': 'native',
 'row_values': {'older_age': 30},
 'evidence_state': 'not_available'}

## 5. Scenario V2：与事实写入相似的 `meta=`，但不是同一事实

Scenario 的产品写法使用普通 `Field`、managed `e_ref` 和 `meta=`。`raw_kind` / `bound` 降为 engine-visible fact semantics；`source` / `sources` 降为封闭的 `ProvenanceRefV1`；`note` / `labels` 是 display/evidence lane。它们都随 Scenario effective world 被捕获，但 Scenario synthetic fact 绝不会被误认为 source ledger 的普通 assertion。

允许的 `meta` key 是闭集：`raw_kind`、`bound`、`source`、`sources`、`note`、`labels`。特别地，`source` 不是裸字符串，也不是 Meander 的 SourceRecord / ACL 载体。

In [6]:
scenario = (
    fg.scenario()
    .set(
        Person.age,
        alice_e_ref,
        35,
        premise_id='scenario:alice-age',
        meta={
            'raw_kind': 'probabilistic',
            'bound': [0.8, 0.8],
            'source': {
                'ref': 'operator:case-42',
                'locator': {'kind': 'line_span', 'line_start': 10, 'line_end': 10},
                'origin_role': 'operator_input',
            },
            'note': 'Run-local age hypothesis',
            'labels': ('demo', 'what-if'),
        },
    )
    .build()
)

assert scenario.operations[0].meta.fact_semantics.point_probability == '0.8'
assert scenario.operations[0].meta.provenance[0].source_ref == 'operator:case-42'
assert fg.fields.get(Person.age, alice_e_ref) == 30  # no ledger mutation


## 6. ProbLog point-probability profile：闭合 semantics 和显式 engine frames

`fg.execution.problog(target=...)` 必须显式调用 `.fact_semantics(identity_probability=True)`；这不是可省略的全局默认值。`for_occurrence(...)`、`for_rule(...)`、`for_choice(...)` 只接受已构造、已归属的 handle 和闭合 marker。Rule-level attachment 与同一 Policy 中该 Rule 的 occurrence-level attachment 不能同时成为一个 lowering slot 的 semantic owner。

下面的 Scenario 假设使 effective row 的点概率为 `0.8`。baseline 保留原始 age=30，概率为 `1`；Native/Soufflé frame 是明确不支持，而不是对 ProbLog 的替代答案。

In [7]:
problog_profile = (
    fg.execution.problog(target=ranked_policy, name='ranked-problog-v2')
    .fact_semantics(identity_probability=True)
    .for_occurrence(older, fg.problog.occurrence_semantics())
    .build()
)

problog_run = (
    fg.query(ranked_policy)
    .bind(older.person, alice)
    .bind(younger.person, bob)
    .select('age', older.age)
    .plan(scenario=scenario, profile=problog_profile)
    .run()
)

baseline_frame = problog_run.baseline.engine_frames[0]
effective_frame = problog_run.effective.engine_frames[0]
assert (baseline_frame.engine, baseline_frame.status) == ('problog', 'succeeded')
assert dict(baseline_frame.observations[0].values)['age'].value == 30
assert baseline_frame.observations[0].point_probability == '1'
assert dict(effective_frame.observations[0].values)['age'].value == 35
assert effective_frame.observations[0].point_probability == '0.8'
assert [(frame.engine, frame.status) for frame in problog_run.effective.engine_frames] == [
    ('problog', 'succeeded'),
    ('native', 'unsupported'),
    ('souffle', 'unsupported'),
]


## 7. 概率 observation、Scenario provenance、materialization 和 Explain 边界

V2 Result View 保留 row/observation digest 与 engine-observed point probability；V2 Explain Data 保留 captured Scenario world、premise id、synthetic origin 和 opaque provenance reference。它还显式记录 declared decimal 到当前 ProbLog float64 materialization 的投影，而不把提交的 decimal 假装成任意精度引擎结果。此 initial ProbLog V2 path 没有捕获 EvidenceGraph，所以 `not_available` 是正确且完整的机器可读状态；绝不能用一个 Native graph 或 prose 来伪造概率证据。

In [8]:
problog_outcome = outcome_from_run_v2(problog_run)
problog_result = problog_outcome.effective
problog_row = problog_result.rows[0]
problog_explain = problog_outcome.explain(problog_row)

scenario_fact = next(
    fact for fact in problog_result.scenario.world.facts
    if fact.origin == 'scenario_synthetic'
)
assert problog_row.point_probability == '0.8'
assert scenario_fact.fact_semantics is not None
assert scenario_fact.fact_semantics.point_probability == '0.8'
assert scenario_fact.provenance[0].source_ref == 'operator:case-42'
assert scenario_fact.premise_ids == ('scenario:alice-age',)
assert problog_explain.evidence.state == 'not_available'
assert problog_explain.evidence.reason_code == 'PROBLOG_V2_EVIDENCE_GRAPH_NOT_CAPTURED'
materialization = problog_result.probability_materialization
assert materialization is not None
assert materialization.model == 'problog_float64_v1'
assert len(materialization.entries) == 1
assert materialization.entries[0].declared_point_probability == '0.8'
assert materialization.entries[0].action == 'emitted'
assert materialization.entries[0].problog_text == materialization.entries[0].float64_text
assert problog_explain.probability_materialization == materialization

{
    'row_identity': problog_row.row_identity_digest,
    'probability': problog_row.point_probability,
    'premise_ids': scenario_fact.premise_ids,
    'source_ref': scenario_fact.provenance[0].source_ref,
    'materialization': {
        'model': materialization.model,
        'declared': materialization.entries[0].declared_point_probability,
        'float64': materialization.entries[0].float64_text,
        'action': materialization.entries[0].action,
    },
    'evidence_state': problog_explain.evidence.state,
}


{'row_identity': 'sha256:7e386b6105b7dfc537fb44f61e5a9c9ad25a2fe9920d1779dbb348b7761b7452',
 'probability': '0.8',
 'premise_ids': ('scenario:alice-age',),
 'source_ref': 'operator:case-42',
 'materialization': {'model': 'problog_float64_v1',
  'declared': '0.8',
  'float64': '0.8',
  'action': 'emitted'},
 'evidence_state': 'not_available'}

## 8. `p=0`：保留 Scenario world，但不伪造 ProbLog fact

`p=0` 是可审计的 Scenario semantic input，不是一个普通 baseline fact，也不是把结果改写成 false。它仍进入 sealed effective world 和 structured materialization capture；当前 ProbLog adapter 明确将其记为 `omitted_zero`，而不生成不能表示的 `0::fact`。由于这个 overlay 覆盖了 Alice 的 age，本 query 没有 effective observation。


In [9]:
zero_scenario = (
    fg.scenario()
    .set(
        Person.age,
        alice_e_ref,
        35,
        premise_id='scenario:zero-age',
        meta={
            'raw_kind': 'probabilistic',
            'bound': ['0', '0'],
            'source': {
                'ref': 'operator:case-42-zero',
                'locator': {'kind': 'line_span', 'line_start': 11, 'line_end': 11},
                'origin_role': 'operator_input',
            },
        },
    )
    .build()
)
zero_run = (
    fg.query(ranked_policy)
    .bind(older.person, alice)
    .bind(younger.person, bob)
    .select('age', older.age)
    .plan(scenario=zero_scenario, profile=problog_profile)
    .run()
)
zero_result = outcome_from_run_v2(zero_run).effective
zero_materialization = zero_result.probability_materialization
assert zero_result.rows == ()
assert zero_materialization is not None
assert len(zero_materialization.entries) == 1
zero_entry = zero_materialization.entries[0]
assert zero_entry.declared_point_probability == '0'
assert zero_entry.action == 'omitted_zero'
assert zero_entry.problog_text is None

{'row_count': len(zero_result.rows), 'zero_action': zero_entry.action}


{'row_count': 0, 'zero_action': 'omitted_zero'}

## 9. `WeightedChoice`：显式排他 stochastic topology，而不是带权 `any`

`WeightedChoice` 在 PolicyBuilder 中显式 author。每个 arm 使用 canonical decimal string，权重精确相加为 `1`，且 `on=` key 必须出现在每一个 arm。它在 V2 ProbLog 中降为一个 annotated disjunction（排他 choice）；普通 `all` / `any` 从不承载 engine parameters 或权重。

本例两个 arm 对 Alice 都成立。排他 AD 的结果是 `0.7 + 0.3 = 1`，而不是把两个分支作为独立 cause 后得到的 `0.79`。

In [10]:
choice_builder = fg.policy_builder(
    'selected_people',
    version='1',
    meta=AssetMeta(name='Selected people', tags=('demo', 'probability')),
)
key = choice_builder.use(person_values, as_='key')
declared = choice_builder.use(person_values, as_='declared')
inferred = choice_builder.use(person_values, as_='inferred')
source_choice = choice_builder.weighted_choice(
    id='eligibility_source',
    on=(key.person,),
    choices=(
        choice_builder.choice('declared', probability='0.7', when=choice_builder.all(declared)),
        choice_builder.choice('inferred', probability='0.3', when=choice_builder.all(inferred)),
    ),
)
weighted_policy = choice_builder.build(choice_builder.all(key, source_choice))

weighted_profile = (
    fg.execution.problog(target=weighted_policy, name='choice-problog-v2')
    .fact_semantics(identity_probability=True)
    .for_choice(source_choice, fg.problog.choice_semantics())
    .build()
)
weighted_run = (
    fg.query(weighted_policy)
    .bind(key.person, alice)
    .select('age', key.age)
    .plan(profile=weighted_profile)
    .run()
)

weighted_frame = weighted_run.effective.engine_frames[0]
assert weighted_policy.requires_v2_profile is True
assert (weighted_frame.engine, weighted_frame.status) == ('problog', 'succeeded')
assert dict(weighted_frame.observations[0].values)['age'].value == 30
assert weighted_frame.observations[0].point_probability == '1'
assert [(frame.engine, frame.status) for frame in weighted_run.effective.engine_frames[1:]] == [
    ('native', 'unsupported'),
    ('souffle', 'unsupported'),
]

weighted_outcome = outcome_from_run_v2(weighted_run)
weighted_result = weighted_outcome.effective
weighted_explain = weighted_outcome.explain(weighted_result.rows[0])
assert weighted_result.choice.attachment_capture.state == 'captured'
assert weighted_result.choice.authored_topology.state == 'captured'
choice_topology = weighted_result.choice.topology
assert choice_topology is not None
assert choice_topology.choice_id == 'eligibility_source'
assert [(arm.arm_id, arm.probability) for arm in choice_topology.arms] == [
    ('declared', '0.7'),
    ('inferred', '0.3'),
]
assert weighted_explain.choice == weighted_result.choice

{
    'choice_id': choice_topology.choice_id,
    'arms': [(arm.arm_id, arm.probability) for arm in choice_topology.arms],
    'selection_key': [
        (item.occurrence_alias, item.port_name)
        for item in choice_topology.selection_key
    ],
}


{'choice_id': 'eligibility_source',
 'arms': [('declared', '0.7'), ('inferred', '0.3')],
 'selection_key': [('key', 'person')]}

## 10. Detached replay

V2 replay consumes only the sealed `EvaluationRunV2` replay payload: captured schema/program/profile/worlds/observations. 它不重新读取 `fg`、ledger、provider、rule registry 或 live asset。`matched` 表示这个固定 payload 在已声明的 engine pins 下重放得到同一 observation frames；它不是远程 runtime attestation，也不声明 proof parity。

In [11]:
scenario_replay = problog_outcome.replay()
choice_replay = weighted_outcome.replay()
assert scenario_replay.status == 'matched'
assert choice_replay.status == 'matched'
assert scenario_replay.proof_parity == 'not_claimed'

{
    'scenario_replay': scenario_replay.status,
    'weighted_choice_replay': choice_replay.status,
    'engine_pin_attestation': scenario_replay.engine_pin_attestation,
}


{'scenario_replay': 'matched',
 'weighted_choice_replay': 'matched',
 'engine_pin_attestation': 'sealed_declared_pins_not_runtime_attested'}

## 11. 已交付的统一点与明确边界

| 主题 | 本教程实际运行的 V2 contract |
| --- | --- |
| Rule / Policy asset | `build_*` 与 `*_builder`；`AssetMeta` 独立于逻辑，asset binding 被 sealed Run 捕获；没有字符串 registry。 |
| 逻辑表达 | 局部 `use(..., as_=...)`、嵌套 `all` / `any`、literal compare、one-hop navigation；不是任意 Python 表达式。 |
| Scenario | run-local fact overlay，严格分离 semantics、provenance、display；不写 ledger，不把 synthetic premise 当 baseline assertion。 |
| 执行参数 | target-scoped V2 profile、闭合 profile attachments、明确资源/capture bounds；不接受泛化 config/engine kwargs。 |
| 概率 | ProbLog engine-observed point probability；Scenario declared decimal 的 `problog_float64_v1` materialization（含 `p=0` omission），以及 exclusive `WeightedChoice` AD；Native/Soufflé V2 probability frame 当前显式 unsupported。 |
| Result / Explain | stable structured V2 views，显式 row target，captured world/provenance/probability；文本只作显示。 |
| Replay | detached sealed-payload replay；不会再次查询 live graph。 |

当前明确不承诺：V2 的 V1 expectations / providers / `EvidenceScopeV1`、任意引擎配置字典、普通 `any` 的概率语义、全局否定或 zero-row negative proof、Meander SourceRecord/ACL/authority 决策、任意 Rule patch Scenario、跨引擎 proof parity，或未捕获 EvidenceGraph 的伪造。

对 AgentPlan 而言，正确的输出是这些受限的声明式资产、occurrence、Policy tree、typed bind/select、Scenario 和 profile pins；FactGraph 计算并封存结果，Meander 再负责资产选择、来源授权、业务解释和行动审批。